# JobPulse AI — Job Market Intelligence & Skill Gap Analyzer

Run this notebook from top to bottom.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn openai

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from openai import OpenAI
from getpass import getpass
pd.set_option("display.max_columns", None)


In [ ]:
df = pd.read_csv("job_postings.csv")
print("Rows:", len(df))
print("Columns:", list(df.columns))
display(df.head())


In [ ]:
print("Missing values:")
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Data types:")
display(df.dtypes)


In [ ]:
df = df.drop_duplicates().copy()
for col in ["title","company","location","education","skills"]:
    df[col] = df[col].astype(str).str.strip()
df["experience_years"] = pd.to_numeric(df["experience_years"], errors="coerce")
df["salary_lpa"] = pd.to_numeric(df["salary_lpa"], errors="coerce")
df = df.dropna(subset=["salary_lpa"])
print("Cleaned rows:", len(df))
display(df.head())


In [ ]:
role_demand = df["title"].value_counts().rename_axis("title").reset_index(name="job_count")
role_salary = df.groupby("title")["salary_lpa"].agg(job_count="count",avg_salary="mean",median_salary="median").sort_values("avg_salary",ascending=False).round(2)
location_salary = df.groupby("location")["salary_lpa"].agg(job_count="count",avg_salary="mean",median_salary="median").sort_values("avg_salary",ascending=False).round(2)
display(role_demand); display(role_salary); display(location_salary)


In [ ]:
skill_rows=[]
for _, row in df.iterrows():
    for skill in row["skills"].split(","):
        skill=skill.strip()
        if skill:
            skill_rows.append({"job_id":row["job_id"],"title":row["title"],"company":row["company"],"location":row["location"],"salary_lpa":row["salary_lpa"],"skill":skill})
skills_df=pd.DataFrame(skill_rows)
skill_demand=skills_df.groupby("skill")["job_id"].nunique().sort_values(ascending=False).reset_index(name="job_count")
skill_demand["demand_pct"]=(skill_demand["job_count"]/df["job_id"].nunique()*100).round(1)
display(skill_demand.head(15))


In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
sns.barplot(data=role_demand,x="job_count",y="title",ax=ax)
ax.set_title("Job Demand by Role"); ax.set_xlabel("Number of postings"); ax.set_ylabel("")
plt.tight_layout(); plt.show()


In [ ]:
top_skills=skill_demand.head(12).sort_values("job_count")
fig,ax=plt.subplots(figsize=(10,6))
sns.barplot(data=top_skills,x="job_count",y="skill",ax=ax)
ax.set_title("Most Requested Skills"); ax.set_xlabel("Unique job postings"); ax.set_ylabel("")
plt.tight_layout(); plt.show()


In [ ]:
fig,ax=plt.subplots(figsize=(10,6))
sns.boxplot(data=df,x="salary_lpa",y="title",ax=ax)
ax.set_title("Salary Distribution by Role"); ax.set_xlabel("Salary (LPA)"); ax.set_ylabel("")
plt.tight_layout(); plt.show()


In [ ]:
conn=sqlite3.connect(":memory:")
df.to_sql("jobs",conn,index=False,if_exists="replace")
skills_df.to_sql("job_skills",conn,index=False,if_exists="replace")
query='''SELECT title, COUNT(*) AS job_count, ROUND(AVG(salary_lpa),2) AS avg_salary_lpa FROM jobs GROUP BY title ORDER BY job_count DESC;'''
display(pd.read_sql_query(query,conn))


In [ ]:
query='''SELECT location, COUNT(*) AS job_count, ROUND(AVG(salary_lpa),2) AS avg_salary_lpa FROM jobs GROUP BY location ORDER BY avg_salary_lpa DESC;'''
display(pd.read_sql_query(query,conn))


In [ ]:
query='''SELECT skill, COUNT(DISTINCT job_id) AS job_count, ROUND(AVG(salary_lpa),2) AS avg_salary_lpa FROM job_skills GROUP BY skill ORDER BY job_count DESC LIMIT 15;'''
display(pd.read_sql_query(query,conn))


In [ ]:
X=df[["title","location","education","experience_years"]]
y=df["salary_lpa"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)
categorical_features=["title","location","education"]
preprocessor=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore"),categorical_features)],remainder="passthrough")
model=Pipeline([("preprocessor",preprocessor),("regressor",RandomForestRegressor(n_estimators=250,max_depth=10,random_state=42))])
model.fit(X_train,y_train)
pred=model.predict(X_test)
mae=mean_absolute_error(y_test,pred)
rmse=np.sqrt(mean_squared_error(y_test,pred))
r2=r2_score(y_test,pred)
print(f"MAE: {mae:.2f} LPA")
print(f"RMSE: {rmse:.2f} LPA")
print(f"R²: {r2:.3f}")


In [ ]:
results=X_test.copy()
results["actual_salary"]=y_test.values
results["predicted_salary"]=np.round(pred,2)
display(results.head(10))


In [ ]:
candidate_skills={"SQL","Excel","Python","Pandas","Power BI"}
target_role="Data Analyst"
role_jobs=df[df["title"]==target_role]
required_skills=set()
for text in role_jobs["skills"]:
    required_skills.update(s.strip() for s in text.split(",") if s.strip())
skill_gap=sorted(required_skills-candidate_skills)
print("Target role:",target_role)
print("Required skills:",sorted(required_skills))
print("Candidate skills:",sorted(candidate_skills))
print("Potential skill gaps:",skill_gap)


In [ ]:
skill_stats=skills_df.groupby("skill").agg(job_count=("job_id","nunique"),avg_salary_lpa=("salary_lpa","mean")).reset_index()
skill_stats["demand_score"]=skill_stats["job_count"]/skill_stats["job_count"].max()*100
skill_stats["salary_score"]=skill_stats["avg_salary_lpa"]/skill_stats["avg_salary_lpa"].max()*100
skill_stats["hiring_signal"]=(0.7*skill_stats["demand_score"]+0.3*skill_stats["salary_score"]).round(1)
display(skill_stats.sort_values("hiring_signal",ascending=False).head(15))


## LLM analysis — requires internet and an OpenAI API key

In [ ]:
api_key=getpass("Enter your OpenAI API key: ")
client=OpenAI(api_key=api_key)
top_skills=skill_stats.sort_values("hiring_signal",ascending=False).head(10)
prompt=f'''You are a data-driven career analyst.
Analyze this job-market snapshot for a Data Analyst candidate.
Top hiring signals:
{top_skills.to_string(index=False)}
Candidate skills:
{sorted(candidate_skills)}
Potential skill gaps:
{skill_gap}
Return:
1. Three market observations.
2. Five skills to prioritize.
3. A short explanation of the skill gaps.
4. A practical 30-day learning plan.
5. One limitation of this dataset.
Use only the supplied information and do not invent statistics.'''
response=client.responses.create(model="gpt-6-luna",input=prompt)
print(response.output_text)


In [ ]:
job_description='''We are looking for a Data Analyst with strong SQL and Python skills.
The candidate should work with Excel, Power BI, dashboards, data cleaning,
business reporting and basic statistical analysis. Experience with cloud data
platforms and machine learning is a plus.'''
analysis_prompt=f'''Extract structured hiring requirements from this job description.
JOB DESCRIPTION:
{job_description}
Return:
- Must-have skills
- Nice-to-have skills
- Core responsibilities
- Suggested interview preparation topics
Do not add skills that are not supported by the text.'''
response=client.responses.create(model="gpt-6-luna",input=analysis_prompt)
print(response.output_text)


In [ ]:
print("JOBPULSE AI")
print("="*50)
print("Jobs analyzed:",df["job_id"].nunique())
print("Companies:",df["company"].nunique())
print("Locations:",df["location"].nunique())
print("Roles:",df["title"].nunique())
print("Skills extracted:",skills_df["skill"].nunique())
print("Target role:",target_role)
print("Potential skill gaps:",", ".join(skill_gap))
print("="*50)
